# Water Audit Queue | Final evaluation

CE 663 · Integrated Water Management

**Research question.** For a multifamily property outside the fitting groups, do floor area and occupancy improve the estimate of annual water-use intensity relative to an overall median and a size-band median?

Run with Python 3, NumPy and pandas. The included CSV files are read from the package’s data folder. Open the notebook from the notebooks folder or run it from the package root. The saved outputs contain the original results. docs/QUICKSTART.md explains the command-line runner.

[NYC Open Data source](https://data.cityofnewyork.us/Environment/NYC-Building-Energy-and-Water-Data-Disclosure-for-/5zyy-y8am/about_data)

# Final evaluation protocol

Freeze this protocol before opening the 2024 water-use outcomes. The population is NYC properties self-classified as multifamily housing. Use the same reporting screen as the pilot, including positive floor area and annual water, occupancy in (0, 100], whole-property metering, no reported water estimate, and an OK complete-year flag. Collapse identical selected rows and exclude unresolved conflicting property-year filings.

Build connected components from the raw metadata across 2022–24 using valid property and parent IDs in one namespace, building IDs, and normalized tax lots. Assign the entire component using SHA-256 of the lowest property ID in that component. Salt `ce663-water-test-v1` reserves hash buckets 0–19 of 100 for the test. A separate salt `ce663-water-val-v1` assigns buckets 0–24 of the remaining components to validation. No outcomes are used to construct the groups or assignments. The 2024 evaluation uses only eligible records in reserved test components. Earlier records from those components are excluded from fitting and calibration. Record split counts and check that no group crosses the boundaries.

Fit the methods on eligible 2022 records from the training components and score eligible 2023 records from the validation components. Compare the overall median, the median in each of four training floor-area quartiles, and ordinary least squares for log water intensity using log floor area and occupancy as the two predictors. Convert regression estimates back to gallons per square foot per year. Select the app's method by lowest validation median absolute error, breaking ties in the stated order. Refit all three methods on the latest eligible 2022–23 record per property in the pooled training and validation components. Score all three on exactly the same frozen 2024 set. The primary metric is median absolute error in US gallons per square foot per year, with mean absolute error secondary. Report the paired difference in median absolute error versus the overall median with a component bootstrap using 2,000 resamples and seed 663.

For the selected method, calculate the 10th and 90th percentiles of observed-to-expected intensity ratios on the 2023 validation set before the refit. Use these thresholds as an empirical comparison band and mark properties above the upper threshold for review. Report the final observed band coverage and review rate. Use only examples from 2023 validation in the offline demo. Report the absence of audit outcomes and the scope of the NYC multifamily benchmark.

In [1]:
from hashlib import sha256
from pathlib import Path
from urllib.parse import urlencode
import io
import re

import numpy as np
import pandas as pd

fields = [
    'report_year', 'property_id', 'parent_property_id', 'nyc_borough_block_and_lot',
    'nyc_building_identification', 'primary_property_type_self', 'primary_property_type',
    'number_of_buildings', 'occupancy', 'property_gfa_self_reported',
    'property_gfa_calculated_1', 'water_use_all_water_sources', 'metered_areas_water',
    'estimated_values_water', 'alert_water_meter_has_less',
]

def read_extract(expected_years, local_names):
    year_list = ','.join(map(str, expected_years))
    query = {
        '$select': ','.join(fields),
        '$where': f"report_year in ({year_list}) and primary_property_type_self = 'Multifamily Housing'",
        '$limit': 100000,
        '$order': 'report_year,property_id,:id',
    }
    url = 'https://data.cityofnewyork.us/resource/5zyy-y8am.csv?' + urlencode(query)
    expected = {str(year) for year in expected_years}
    for folder in (Path.cwd() / 'data', Path.cwd().parent / 'data'):
        for name in local_names:
            path = folder / name
            if not path.is_file():
                continue
            data = path.read_bytes()
            try:
                frame = pd.read_csv(io.BytesIO(data), dtype='string', usecols=fields)
            except ValueError:
                continue
            if set(frame.report_year.unique()) == expected and frame.primary_property_type_self.eq('Multifamily Housing').all():
                return frame, sha256(data).hexdigest()
    raise FileNotFoundError(f'Place the included CSV extracts in the package data folder. Source query {url}')


early, early_sha = read_extract([2022, 2023], ['nyc_multifamily_2022_2023.csv'])
future, future_sha = read_extract([2024], ['nyc_multifamily_2024.csv'])
raw = pd.concat([early, future], ignore_index=True)
print('2022–23 extract SHA-256:', early_sha)
print('2024 extract SHA-256:', future_sha)
print('Raw multifamily records by year:', raw.report_year.value_counts().sort_index().to_dict())

2022–23 extract SHA-256: a2ff47070d532c78e9d2116cb73682246bd8c8b6c6e76308b741bea5686381cf
2024 extract SHA-256: b7e9f2348d6656ca66057f0e28f4efe5aec1d207f4498105f56c3aa21cd36a35
Raw multifamily records by year: {'2022': 19732, '2023': 22219, '2024': 26745}


## Lock the split using identifiers only

Each connected component gets one deterministic assignment. A tax lot can link different buildings, so grouping by lot is deliberately conservative. Unrecorded or unrecognized identifiers can still leave related properties unlinked.

In [2]:
def property_keys(row):
    keys = set()
    for value in (row.property_id, row.parent_property_id):
        if pd.notna(value) and re.fullmatch(r'[1-9]\d*', value.strip()):
            keys.add(('property', value.strip()))
    if pd.notna(row.nyc_building_identification):
        for number in re.findall(r'\b[1-5]\d{6}\b', row.nyc_building_identification):
            if number[1:] != '000000':
                keys.add(('building', number))
    if pd.notna(row.nyc_borough_block_and_lot):
        lots = row.nyc_borough_block_and_lot
        for borough, block, lot in re.findall(r'\b([1-5])\s*-\s*(\d{1,5})\s*-\s*(\d{1,4})\b', lots):
            if int(block) and int(lot):
                keys.add(('lot', f'{borough}{int(block):05d}{int(lot):04d}'))
        for number in re.findall(r'\b[1-5]\d{9}\b', lots):
            if int(number[1:6]) and int(number[6:]):
                keys.add(('lot', number))
    return keys


def root(parents, item):
    while parents[item] != item:
        parents[item] = parents[parents[item]]
        item = parents[item]
    return item


def bucket(salt, group):
    return int.from_bytes(sha256(f'{salt}:{group}'.encode()).digest(), 'big') % 100


metadata = raw[['property_id', 'parent_property_id', 'nyc_building_identification', 'nyc_borough_block_and_lot']]
metadata = metadata.loc[metadata.property_id.str.fullmatch(r'[1-9]\d*', na=False)].drop_duplicates()
parents = {int(pid): int(pid) for pid in metadata.property_id.unique()}
key_owners = {}
has_site_key = set()
for row in metadata.itertuples(index=False):
    pid = int(row.property_id)
    keys = property_keys(row)
    if any(kind in {'building', 'lot'} for kind, _ in keys):
        has_site_key.add(row.property_id)
    for key in keys:
        owner = key_owners.setdefault(key, pid)
        left, right = sorted((root(parents, pid), root(parents, owner)))
        parents[right] = left

group_by_property = {str(pid): root(parents, pid) for pid in parents}
group_ids = sorted(set(group_by_property.values()))
split_by_group = {}
for group in group_ids:
    if bucket('ce663-water-test-v1', group) < 20:
        split_by_group[group] = 'test'
    elif bucket('ce663-water-val-v1', group) < 25:
        split_by_group[group] = 'validation'
    else:
        split_by_group[group] = 'train'

assert all(group <= int(pid) and group_by_property[str(group)] == group
           for pid, group in group_by_property.items())
split_manifest = '\n'.join(f'{pid},{group},{split_by_group[group]}'
                           for pid, group in sorted(group_by_property.items(), key=lambda item: int(item[0])))
print('Frozen split SHA-256:', sha256(split_manifest.encode()).hexdigest())
print('Groups across raw identifier metadata:', pd.Series(split_by_group).value_counts().to_dict())

Frozen split SHA-256: f19cf84a59562a98d3ccda59585afee72aad3d2803603ebe8724e52eb55ca693
Groups across raw identifier metadata: {'train': 11412, 'test': 3756, 'validation': 3678}


In [3]:
def screen(records):
    unique = records.drop_duplicates().copy()
    identified = unique.property_id.isin(group_by_property)
    conflicts = identified & unique.duplicated(['report_year', 'property_id'], keep=False)
    frame = unique.loc[identified & ~conflicts].copy()
    numeric = ['occupancy', 'property_gfa_self_reported', 'water_use_all_water_sources']
    for column in numeric:
        frame[column] = pd.to_numeric(frame[column], errors='coerce')
    eligible = (
        frame.occupancy.gt(0) & frame.occupancy.le(100)
        & frame.property_gfa_self_reported.gt(0) & frame.water_use_all_water_sources.gt(0)
        & np.isfinite(frame.property_gfa_self_reported) & np.isfinite(frame.water_use_all_water_sources)
        & frame.metered_areas_water.eq('Whole Property') & frame.estimated_values_water.eq('No')
        & frame.alert_water_meter_has_less.eq('Ok')
    )
    frame = frame.loc[eligible].copy()
    frame['group'] = frame.property_id.map(group_by_property)
    frame['split'] = frame.group.map(split_by_group)
    frame['wui'] = 1000 * frame.water_use_all_water_sources / frame.property_gfa_self_reported
    return frame, len(records) - len(unique), int(conflicts.sum())


history, early_repeats, early_conflicts = screen(early)
train = history.loc[history.report_year.eq('2022') & history.split.eq('train')].copy()
validation = history.loc[history.report_year.eq('2023') & history.split.eq('validation')].copy()
final_fit = (history.loc[history.split.ne('test')].sort_values('report_year')
             .drop_duplicates('property_id', keep='last').copy())
assert set(train.group).isdisjoint(validation.group)
print('2022–23 exact repeats collapsed:', early_repeats)
print('2022–23 conflicting property-year rows excluded:', early_conflicts)
for name, frame in [('2022 initial train', train), ('2023 validation', validation), ('2022–23 final fit', final_fit)]:
    print(f'{name}: {len(frame):,} properties across {frame.group.nunique():,} groups')

2022–23 exact repeats collapsed: 409
2022–23 conflicting property-year rows excluded: 368
2022 initial train: 8,430 properties across 7,823 groups
2023 validation: 2,549 properties across 2,325 groups
2022–23 final fit: 13,054 properties across 11,246 groups


## Fit the three benchmarks and select the demo method

The regression learns log intensity from log floor area and occupancy as a fraction. The exponential maps its predictions back to the original units. Each method uses the same properties and targets. The empirical comparison band multiplies the expected value by the lower and upper ratios learned from validation data.

In [4]:
labels = ['Overall median', 'Size-band median', 'Area + occupancy regression']


def features(frame):
    area = frame.property_gfa_self_reported.to_numpy(dtype=float)
    occupancy = frame.occupancy.to_numpy(dtype=float)
    return np.column_stack([np.ones(len(frame)), np.log(area), occupancy / 100])


def fit(frame):
    area = frame.property_gfa_self_reported.to_numpy(dtype=float)
    intensity = frame.wui.to_numpy(dtype=float)
    cuts = np.quantile(area, [0.25, 0.50, 0.75])
    bands = np.searchsorted(cuts, area, side='left')
    medians = [float(np.median(intensity[bands == band])) for band in range(4)]
    coefficients = np.linalg.lstsq(features(frame), np.log(intensity), rcond=None)[0]
    return {
        'overall': float(np.median(intensity)), 'cuts': cuts.tolist(),
        'bandMedians': medians, 'coefficients': coefficients.tolist(),
        'areaReference': np.quantile(area, [0.01, 0.99]).tolist(),
    }


def predict(model, frame):
    areas = frame.property_gfa_self_reported.to_numpy(dtype=float)
    return np.column_stack([
        np.full(len(frame), model['overall']),
        np.asarray(model['bandMedians'])[np.searchsorted(model['cuts'], areas, side='left')],
        np.exp(features(frame) @ model['coefficients']),
    ])


initial_model = fit(train)
validation_predictions = predict(initial_model, validation)
validation_errors = np.abs(validation.wui.to_numpy(dtype=float)[:, None] - validation_predictions)
validation_median_errors = np.median(validation_errors, axis=0)
selected = int(np.argmin(validation_median_errors))
validation_ratios = validation.wui.to_numpy(dtype=float) / validation_predictions[:, selected]
band_low, band_high = np.quantile(validation_ratios, [0.10, 0.90])
model = fit(final_fit)

print(pd.DataFrame({'Method': labels, '2023 median abs. error': validation_median_errors}).round(2).to_string(index=False))
print('Selected before 2024 scoring:', labels[selected])
print(f'2023 observed-to-expected ratio band: {band_low:.3f}× to {band_high:.3f}×')

                     Method  2023 median abs. error
             Overall median                   23.89
           Size-band median                   23.40
Area + occupancy regression                   22.65
Selected before 2024 scoring: Area + occupancy regression
2023 observed-to-expected ratio band: 0.434× to 2.218×


## Score the frozen 2024 test

Errors are in US gallons per square foot per year. Lower error is better. Bootstrap intervals compare each method's median absolute error with the overall median on the same resampled property groups. The band is assessed on this test set after calibration on 2023.

In [5]:
future_screened, future_repeats, future_conflicts = screen(future)
test = future_screened.loc[future_screened.split.eq('test')].copy()
assert test.property_id.is_unique
assert set(test.group).isdisjoint(final_fit.group)
assert set(test.group).isdisjoint(validation.group)

predictions = predict(model, test)
observed = test.wui.to_numpy(dtype=float)
errors = np.abs(observed[:, None] - predictions)
medians = np.median(errors, axis=0)
means = np.mean(errors, axis=0)

group_indices = list(test.groupby('group').indices.values())
rng = np.random.default_rng(663)
changes = np.empty((2000, 2))
for sample in range(len(changes)):
    drawn = rng.integers(len(group_indices), size=len(group_indices))
    row_indices = np.concatenate([group_indices[index] for index in drawn])
    bootstrap_medians = np.median(errors[row_indices], axis=0)
    changes[sample] = bootstrap_medians[1:] - bootstrap_medians[0]
intervals = np.quantile(changes, [0.025, 0.975], axis=0).T

ratios = observed / predictions[:, selected]
coverage = float(np.mean((ratios >= band_low) & (ratios <= band_high)))
review_rate = float(np.mean(ratios > band_high))
results = pd.DataFrame({
    'Method': labels, '2023 median AE': validation_median_errors, '2024 median AE': medians,
    '2024 mean AE': means, 'Median AE difference': medians - medians[0],
    '95% interval vs overall': ['Reference', *[f'[{lo:.2f}, {hi:.2f}]' for lo, hi in intervals]],
})

print(f'2024 screen: {len(future_screened):,} eligible properties, {future_repeats} exact repeats, {future_conflicts} conflicting rows')
print(f'2024 test: {len(test):,} properties across {test.group.nunique():,} groups')
print(f'Largest test group: {max(map(len, group_indices))} properties')
print(f'Test properties without a valid building ID or lot in metadata: {(~test.property_id.isin(has_site_key)).sum()}')
print(results.round(2).to_string(index=False))
print(f'Comparison band coverage: {coverage:.1%}')
print(f'Sent for review: {review_rate:.1%}')

2024 screen: 11,966 eligible properties, 0 exact repeats, 0 conflicting rows
2024 test: 2,394 properties across 2,160 groups
Largest test group: 10 properties
Test properties without a valid building ID or lot in metadata: 1
                     Method  2023 median AE  2024 median AE  2024 mean AE  Median AE difference 95% interval vs overall
             Overall median           23.89           23.11         42.21                  0.00               Reference
           Size-band median           23.40           22.77         41.03                 -0.34           [-1.41, 0.67]
Area + occupancy regression           22.65           21.22         41.30                 -1.89          [-2.68, -0.81]
Comparison band coverage: 78.9%
Sent for review: 11.4%


## Interpretation for the term paper

The area-and-occupancy regression selected on 2023 validation data reduced median absolute error on the 2024 holdout from 23.11 to 21.22 gal/ft²/year. The paired difference was −1.89 gal/ft²/year, with a 95% group-bootstrap interval of −2.68 to −0.81. The size-band baseline had the lowest mean absolute error (41.03 versus 41.30 for regression). The fitted model improved the typical error modestly, while large errors remained important.

The empirical comparison band covered 78.9% of held-out observations, and 11.4% exceeded its upper threshold. These are observed rates across this NYC multifamily holdout. The band describes reported water intensity, and the dataset supplies no confirmed leaks or audit outcomes. Transfer to IIT Kanpur would require local benchmarking data and a separate check of property types, meter coverage, and water-use patterns.